In [1]:
import sys

assert sys.version_info >= (3, 10)

In [2]:
import torch
from packaging.version import Version

assert Version(torch.__version__) >= Version("2.6.0")

In [3]:
import matplotlib.pyplot as plt

plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

In [4]:
from pathlib import Path

IMAGES_PATH = Path() / "images" / "Poisson_eqn_with_Robinbc"
IMAGES_PATH.mkdir(parents=True, exist_ok=True)

def save_fig(fig_id, fig_extension="png", tight_layout=True, resolution=300):
    path = IMAGES_PATH / f"{fig_id}.{fig_extension}"
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension, dpi=resolution)

In [5]:
import deepxde as dde
from deepxde import utils
import numpy as np

dde.config.set_random_seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

Using backend: pytorch
Other supported backends: tensorflow.compat.v1, tensorflow, jax, paddle.
paddle supports more examples now and is recommended.


Setting the backend

In [6]:
dde.config.set_default_float("float64")
print(f"Backend: {dde.backend.backend_name}")

Set the default float type to float64
Backend: pytorch


Define Exact solution for validation

In [7]:
def exact_solution(x):
    return (x + 1) ** 2

Defining the Domain geometry

In [8]:
geom = dde.geometry.Interval(-1,1)


In [14]:
def pde(x, y):
    dy_xx = dde.grad.hessian(y, x, i=0, j=0)
    return dy_xx - 2

def boundary_left(x, on_boundary):
    return on_boundary and np.isclose(x[0], -1)

def boundary_right(x, on_boundary):
    return on_boundary and np.isclose(x[0], 1)




Creating a function for Robin Boundary Condition

In [15]:
bc_left = dde.icbc.DirichletBC(geom, lambda x: 0, boundary_left)

def robin_bc(x, y, X):
    dy_dx = dde.grad.jacobian(y, x, i=0, j=0)
    return dy_dx - y

bc_right = dde.icbc.OperatorBC(
    geom, robin_bc, boundary_right
)

creating some training data

In [16]:
observe_x = np.linspace(1, 10, 35).reshape(-1, 1)
observe_y = exact_solution(observe_x)
noise = 0.1 * np.random.randn(35, 1)
observe_y = observe_y + noise
observe = dde.icbc.PointSetBC(observe_x, observe_y, component=0)

Combining all data

In [17]:
data = dde.data.PDE(
    geom, pde,[bc_left, bc_right, observe],
    num_domain=3000, num_boundary=200,
    num_test=500, anchors=observe_x
)

Build a neural network and create a model

In [18]:
net = dde.nn.FNN([1, 256, 128, 64, 1], "tanh", "Glorot uniform")

model = dde.Model(data, net)

Setting the loss weights instead we can also use adaptive weights using a class

In [19]:
loss_weights = [1.0, 40.0, 1.0, 500.0]

Train the model using
    1. Adam optimizers
    2. L-BFGS optimizer(optional)

In [20]:
print("\nStage 1: Adam Optimizer")
model.compile("adam", lr=0.001, loss_weights=loss_weights,
              decay=("inverse time", 1000, 0.1))
losshistory, train_state = model.train(iterations=2000, display_every=200)

dde.optimizers.config.set_LBFGS_options(maxiter=200)
model.compile("L-BFGS", loss_weights = loss_weights)
losshistory, train_state = model.train(display_every=50)


Stage 1: Adam Optimizer
Compiling model...
'compile' took 1.909576 s

Training model...



/home/ziaur/ziazh/lib/python3.14/site-packages/torch/autograd/graph.py:882: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:370.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Step      Train loss                                  Test loss                                   Test metric
0         [4.00e+00, 2.39e-01, 2.82e-05, 1.83e+06]    [4.00e+00, 2.39e-01, 2.82e-05, 1.83e+06]    []  
200       [5.67e+01, 1.21e+02, 2.06e+02, 1.00e+06]    [5.99e+01, 1.21e+02, 2.06e+02, 1.00e+06]    []  
400       [5.12e+00, 1.10e+01, 1.16e+00, 6.93e+05]    [4.99e+00, 1.10e+01, 1.16e+00, 6.93e+05]    []  
600       [6.81e+00, 4.49e-01, 3.19e-01, 4.91e+05]    [6.47e+00, 4.49e-01, 3.19e-01, 4.91e+05]    []  
800       [8.22e+00, 6.81e-02, 1.78e-01, 3.53e+05]    [7.70e+00, 6.81e-02, 1.78e-01, 3.53e+05]    []  
1000      [8.46e+00, 2.06e-02, 6.70e-02, 2.57e+05]    [7.87e+00, 2.06e-02, 6.70e-02, 2.57e+05]    []  
1200      [8.24e+00, 8.86e-03, 1.76e-02, 1.89e+05]    [7.12e+00, 8.86e-03, 1.76e-02, 1.89e+05]    []  
1400      [6.69e+00, 4.10e-03, 7.37e-04, 1.40e+05]    [6.01e+00, 4.10e-03, 7.37e-04, 1.40e+05]    []  
1600      [5.75e+00, 1.51e-03, 1.89e-03, 1.04e+05]    [4.86e+00, 1

Create some test points

In [21]:
test_points  = 15
x_test  = np.linspace(0, 10, test_points).reshape(-1, 1)
y_pred = model.predict(x_test)
y_exact= exact_solution(x_test)
noise = 0.03 * np.random.randn(15, 1)
y_test = y_exact + noise

Evaluate the results

In [26]:
absolute_error = np.abs(y_test - y_pred)
relative_error = absolute_error / (np.abs(y_test) + 1e-10)
l2_error = np.linalg.norm(y_test - y_pred) / np.linalg.norm(y_test)

print("\n" + "="*60)
print("📊 PERFORMANCE METRICS")
print("="*60)
print(f"L2 Relative Error:      {l2_error:.6f}")
print(f"Max Absolute Error:     {np.max(absolute_error):.6f}")
print(f"Mean Absolute Error:    {np.mean(absolute_error):.6f}")



📊 PERFORMANCE METRICS
L2 Relative Error:      0.000963
Max Absolute Error:     0.126402
Mean Absolute Error:    0.047869


Predicting the results at the boundaries

In [25]:
u_at_minus1 = model.predict(np.array([[-1.0]]))[0, 0]
u_at_plus1 = model.predict(np.array([[1.0]]))[0, 0]

x_right = np.array([[1.0]])
du_dx_at_1 = model.predict(
    x_right, operator=lambda x, y:dde.grad.jacobian(y, x, i=0, j=0)[0, 0]
)
u_at_1 = model.predict(x_right)[0, 0]
robin_residual = du_dx_at_1 - u_at_1

In [27]:
print("\n" + "="*60)
print("📊 PERFORMANCE METRICS")
print("="*60)
print(f"L2 Relative Error:      {l2_error:.6f}")
print(f"Max Absolute Error:     {np.max(absolute_error):.6f}")
print(f"Mean Absolute Error:    {np.mean(absolute_error):.6f}")

print("\n📍 BOUNDARY CONDITION CHECK")
print("-"*60)
print(f"u(-1) = {u_at_minus1:.6f}  (Target: 0.0)")
print(f"u(1) = {u_at_1:.6f}  (Expected: ~4.0)")
print(f"du/dx(1) = {du_dx_at_1:.6f}  (Expected: ~4.0)")
print(f"Robin BC: du/dx - u = {robin_residual:.6f}  (Target: 0.0)")
print("="*60)


📊 PERFORMANCE METRICS
L2 Relative Error:      0.000963
Max Absolute Error:     0.126402
Mean Absolute Error:    0.047869

📍 BOUNDARY CONDITION CHECK
------------------------------------------------------------
u(-1) = 0.000472  (Target: 0.0)
u(1) = 3.889274  (Expected: ~4.0)
du/dx(1) = 4.006076  (Expected: ~4.0)
Robin BC: du/dx - u = 0.116802  (Target: 0.0)


DeepXDE computes the exact losses used during training


In [28]:
final_train_losses = model.losshistory.loss_train[-1]
L_pde, L_bc_l, L_bc_r, L_data = final_train_losses

In [29]:
print("\n" + "="*60)
print("📉 FINAL TRAINING LOSSES (Unweighted - REAL values used by DeepXDE)")
print("="*60)
print(f"L_PDE      : {L_pde:.6e}")
print(f"L_BC_left  : {L_bc_l:.6e}")
print(f"L_BC_right : {L_bc_r:.6e}")
print(f"L_data     : {L_data:.6e}")


📉 FINAL TRAINING LOSSES (Unweighted - REAL values used by DeepXDE)
L_PDE      : 4.137545e-02
L_BC_left  : 8.911950e-06
L_BC_right : 1.364278e-02
L_data     : 1.516950e+00
